In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.label.label_generator import LabelGenerator

lg = LabelGenerator(time_start=1430, time_end=1457, price_type="twap")
price_path, label_path = lg.generate_and_save()
print(f"价格表: {price_path}")
print(f"Label:  {label_path}")

In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.label.label_generator import generate_and_save_open0935_1000_label

# 全市场分钟数据范围内生成并落盘 (与 TWAP Label 同目录 outputs/labels/)
# 可选: start_date=20210104, end_date=20251231 缩小范围以加快调试
# save_price_tables=True 会额外保存 09:35/10:00 两张 open 宽表, 便于回测直接读价
open_label_path = generate_and_save_open0935_1000_label(save_price_tables=True)
print(f"OPEN0935->次日10:00 Label: {open_label_path}")


In [ ]:
import sys, logging
sys.path.insert(0, '/root/autodl-tmp')
logging.basicConfig(level=logging.INFO)

from Strategy import config
from Strategy.factor.daily_factor_library import DailyFactorLibraryAdapter

# 日频因子需覆盖 训练+验证+样本外 至数据末，否则 val / oos / 打分缺特征（见 config 注释）
# end_date 不传 = 用 Daily_data 中最后交易日；仅调试可写 end_date=config.TRAIN_END
adapter = DailyFactorLibraryAdapter()
saved = adapter.compute_and_save_all(
    start_date=config.TRAIN_START,
    # end_date=config.TRAIN_END,  # 仅训练内调试时取消注释
)
print(f"已保存 {len(saved)} 个因子")

In [ ]:
import sys, logging
sys.path.insert(0, '/root/autodl-tmp')
logging.basicConfig(level=logging.INFO)

from Strategy.factor.factor_base import FactorRegistry
import Strategy.factor.minute_derived_factors   # 触发注册
import Strategy.factor.custom_factors           # 触发注册
 
FactorRegistry.compute_all()

In [ ]:
import sys, logging
sys.path.insert(0, '/root/autodl-tmp')
logging.basicConfig(level=logging.INFO)

from Strategy.factor.minute_derived_factors import JumpVariationFactor

jv = JumpVariationFactor()
jv.compute_and_save()  # 输出 RV / RVC / RVJ 等 13 个因子

In [1]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label
from Strategy.model.trainer import build_panel, split_panel

# 加载
factor_dict = load_all_factors()          # 自动读取 outputs/factors/ 下所有 .fea
label_df = load_label("OPEN0935_1000")

# 拼接长表 Panel
panel = build_panel(factor_dict, label_df)
print(panel.shape)   # (样本数, 因子数+3)

# 按时间划分 (config.py 中已定义日期)
# Train: 2021-01-01 ~ 2023-08-01
# Val:   2023-09-01 ~ 2024-09-01
# OOS:   2024-09-01 之后
# train, val, oos = split_panel(panel)

(6660324, 137)


In [2]:
from Strategy.model.trainer import XGBTrainer
from Strategy.model.rolling_trainer import RollingTrainer

# ── 方案 A: 滚动训练 (推荐, 减少过拟合) ──
rt = RollingTrainer(
    model_class=XGBTrainer,
    model_kwargs={
        "use_wpcc": True,          # WPCC 损失直接优化 IC
        "label_winsorize_sigma": 3.0,
    },
    train_start="2021-01-01",
    first_train_months=9,          # 第一个 Fold: 2021-01~09
    val_months=3,                  # 验证窗口: 3 个月
)
rt.train_all_folds(panel)          # panel 来自上一个 Cell

# 查看每个 Fold 的 IC
print(rt.fold_ic_report())

# XGBoost 特征重要性汇总
fi = rt.get_feature_importance()
if fi is not None:
    print(fi.head(10))

# 保存滚动模型
rt.save_model()

# ── 生成打分宽表 ──
from Strategy.model.scorer import mask_scores_by_current_price
from Strategy.data_io.saver import save_wide_table
from Strategy import config

score_wide = rt.score_all(panel, normalize=True)
score_wide = mask_scores_by_current_price(score_wide, label_tag="OPEN0935_1000")

# 保存打分
save_wide_table(score_wide, config.SCORE_OUTPUT_DIR / "SCORE_xgb_OPEN0935_1000.fea")
print(f"打分已保存, shape={score_wide.shape}")

[0]	train-rmse:0.49993	train-wpcc:-0.06722	val-rmse:0.49951	val-wpcc:-0.02958
[81]	train-rmse:0.49992	train-wpcc:-0.20416	val-rmse:0.49950	val-wpcc:-0.01311
[0]	train-rmse:0.49982	train-wpcc:-0.06692	val-rmse:0.50331	val-wpcc:-0.04571
[82]	train-rmse:0.49981	train-wpcc:-0.18194	val-rmse:0.50331	val-wpcc:-0.03620
[0]	train-rmse:0.50055	train-wpcc:-0.06600	val-rmse:0.50068	val-wpcc:-0.03664
[87]	train-rmse:0.50054	train-wpcc:-0.17342	val-rmse:0.50068	val-wpcc:-0.04331
[0]	train-rmse:0.50057	train-wpcc:-0.06179	val-rmse:0.50186	val-wpcc:-0.04791
[100]	train-rmse:0.50057	train-wpcc:-0.17009	val-rmse:0.50186	val-wpcc:-0.04173
[102]	train-rmse:0.50057	train-wpcc:-0.17133	val-rmse:0.50186	val-wpcc:-0.04222
[0]	train-rmse:0.50078	train-wpcc:-0.05843	val-rmse:0.50025	val-wpcc:-0.02848
[91]	train-rmse:0.50078	train-wpcc:-0.15512	val-rmse:0.50025	val-wpcc:-0.04244
[0]	train-rmse:0.50071	train-wpcc:-0.05406	val-rmse:0.49785	val-wpcc:-0.04685
[85]	train-rmse:0.50071	train-wpcc:-0.14414	val-rmse:0.4

In [ ]:
import sys
import logging

sys.path.insert(0, '/root/autodl-tmp')
logging.basicConfig(level=logging.INFO, force=True)

from Strategy.model.transformer_trainer import TransformerTrainer
from Strategy.model.rolling_trainer import RollingTrainer
from Strategy.model.scorer import mask_scores_by_current_price
from Strategy.data_io.saver import save_wide_table
from Strategy import config

# ── Transformer 滚动训练 ──
rt_tf = RollingTrainer(
    model_class=TransformerTrainer,
    model_kwargs={
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "d_ff": 128,
        "dropout": 0.15,
        "epochs": 50,
        "lr": 5e-4,
        "weight_decay": 0.01,
        "batch_size": 16,              # 5090 32GB 安全值
        "early_stopping_patience": 8,
        "device": "cuda",
        "use_amp": False,
        "label_winsorize_sigma": 0,
    },
    train_start="2021-01-01",
    first_train_months=9,
    val_months=3,
)

# panel 来自前面 build_panel + split_panel 之前的完整 panel
rt_tf.train_all_folds(panel)

print(rt_tf.fold_ic_report())

rt_tf.save_model(config.SCORE_OUTPUT_DIR / "rolling_transformer_model.pkl")

# ── 生成 Transformer 打分宽表 ──
LABEL_TAG = "OPEN0935_1000"

score_wide_tf = rt_tf.score_all(panel, normalize=True)
score_wide_tf = mask_scores_by_current_price(score_wide_tf, label_tag=LABEL_TAG)

save_wide_table(score_wide_tf, config.SCORE_OUTPUT_DIR / f"SCORE_transformer_{LABEL_TAG}.fea")
print(f"Transformer 打分已保存, shape={score_wide_tf.shape}")


In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.model.scorer import load_scores
from Strategy.model.ensemble_scorer import (
    compute_score_correlation,
    select_ensemble_models,
    ensemble_scores,
)
from Strategy import config

LABEL_TAG = "OPEN0935_1000"

# ── 1. 加载各模型打分 ──
score_dfs = {
    "xgb":         load_scores("xgb", LABEL_TAG),
    "transformer": load_scores("transformer", LABEL_TAG),
}

# ── 2. 查看两两打分相关性 ──
corr = compute_score_correlation(score_dfs)
print("模型打分截面相关性矩阵:")
print(corr)

# ── 3. 自动筛选入选模型 (ICIR + 去相关) ──
selected = select_ensemble_models(
    score_dfs,
    ic_summaries=None,
    max_pairwise_corr=0.8,
)
print(f"入选模型: {selected}")

# ── 4. 集成打分 = Z-Score 标准化后简单平均 ──
ensemble_df = ensemble_scores(
    score_dfs,
    selected_models=selected,
    label_tag=LABEL_TAG,
    save=True,                    # 自动保存到 SCORE_ensemble_OPEN0935_1000.fea
    output_dir=config.SCORE_OUTPUT_DIR,
)
print(f"集成打分: shape={ensemble_df.shape}")


In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.backtest.quick_backtest import run_quick_backtest
from Strategy.model.scorer import load_scores
from Strategy.label.label_generator import load_label
from Strategy import config

LABEL_TAG = "OPEN0935_1000"
label_df  = load_label(LABEL_TAG)

TOP_KS  = (20, 50, 100)
TAIL_KS = (20, 50, 100)

models_to_test = ["xgb", "transformer", "ensemble"]

for model_name in models_to_test:
    print(f"\n{'='*50}\n开始回测: {model_name}\n{'='*50}")
    
    try:
        score_df = load_scores(model_name, LABEL_TAG)
    except FileNotFoundError:
        print(f"找不到 {model_name} 的打分文件，跳过...")
        continue
        
    quick_ret = run_quick_backtest(
        score_df=score_df,
        label_df=label_df,
        n_groups=20,
        output_dir=config.BT_RESULT_DIR / model_name,  # 各模型分开建文件夹
        start_date=config.VAL_START,
        top_ks=TOP_KS,
        tail_ks=TAIL_KS,
        run_ic=True,
        ic_rolling_window=20,
        title=f"{model_name.upper()} | {LABEL_TAG}",
    )
    print(f"{model_name} 回测完成。图表和 IC 分析已保存至: {config.BT_RESULT_DIR / model_name}")


In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.backtest.quick_backtest import run_quick_backtest
from Strategy.model.scorer import load_scores
from Strategy.label.label_generator import load_label
from Strategy import config

score_df = load_scores("xgb", "OPEN0935_1000")
label_df = load_label("OPEN0935_1000")

# 可自行调整想看的 topN 曲线, 例如 (10, 20, 50, 100, 200)
TOP_KS  = (20, 50, 100)
TAIL_KS = (20, 50, 100)

quick_ret = run_quick_backtest(
    score_df=score_df,
    label_df=label_df,
    n_groups=20,
    output_dir=config.BT_RESULT_DIR,
    start_date=config.VAL_START,
    top_ks=TOP_KS,
    tail_ks=TAIL_KS,              # ← 新增: 最低分后 K 只
    run_ic=True,                  # ← 新增: 同步运行 IC/Rank IC 分析
    ic_rolling_window=20,         # ← 新增: IC 滚动均线窗口
)


In [ ]:
# import sys
# sys.path.insert(0, '/root/autodl-tmp')

# from Strategy.backtest.event_backtest import run_batch_event_backtest
# from Strategy.model.scorer import load_scores
# from Strategy import config

# score_df = load_scores("xgb", "OPEN0935_1000")

# # 一次性批量跑多个 TopN / TailN / 分位组，输出汇总 NAV 对比图
# results = run_batch_event_backtest(
#     score_df=score_df,
#     top_ns=(20, 50, 100),          # 多头: 最高分前 N 只
#     tail_ns=(20, 50, 100),         # 空头参考: 最低分后 N 只
#     quantile_groups=(1, 20),       # 可选: 同时跑第1组(最高) 和 第20组(最低)
#     n_quantile_groups=20,
#     start_date=config.VAL_START,
#     rebalance_freq=1,
#     frictionless=True,             # 与 quick_backtest 无佣金假设对齐; 实盘改 False
#     output_dir=config.BT_RESULT_DIR,
#     title="Event Backtest: TopN vs TailN vs Quantile Groups",
# )
